# 05 Lichtkurve & Periode

**Eingabe**: `messtabelle.csv` aus Notebook 04  
**Ausgabe**: `zeitreihe.csv` (BJD_TDB, Magnitude, Fehler, Qualitätsspalten), Lichtkurve + Phasenkurve

Schritte: 1 Messtabelle laden · 2 Differenzphotometrie gegen das Comp-Ensemble · 3 Lomb-Scargle-Periode · 4 Phasenkurve · 5 Export

## Schritt 1 — Messtabelle und Standort

Der Standort geht in BJD_TDB und die Szintillation ein. Er kommt aus dem **Bild-Header** (LCO: `SITE`), sonst aus
Notebook 01 (dein Standort).

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle

from varphot import light_curve, read_frame, read_measurements, site_from_header, stars_from_field_config
from varphot.observer import data_dir, load_config, all_sites
from varphot.testdata import testdata_dir

TAG     = os.environ.get("VARPHOT_TAG", "delta_cep_lco_0m4_11")
RESULTS = (data_dir() if data_dir().is_dir() else Path("../data")) / "results" / TAG
FIELD_CONFIG = Path(os.environ.get("VARPHOT_FIELD_CONFIG", "../data/fields/del_Cep_lco_0m4_field_config.json"))
FRAMES_DIR = Path(os.environ.get("VARPHOT_FRAMES_DIR") or testdata_dir() / "lco-dcep").expanduser()   # Download: tools/download_testdata.py
APERTURE_M = float(os.environ.get("VARPHOT_APERTURE_M", 0.4))      # Teleskopöffnung für die Szintillation

MT = read_measurements(RESULTS / "messtabelle.csv")
MT["frame_flags"] = MT["frame_flags"].fillna("")
record = json.loads(FIELD_CONFIG.read_text())
STARS = stars_from_field_config(record)
STARS["role"] = [{"g1": "check"}.get(i, r) for i, r in zip(STARS["id"], STARS["role"])]
STARS["id"] = STARS["id"].astype(str)

first = FRAMES_DIR / MT["file"].iloc[0]
SITE = site_from_header(read_frame(first)[1]) if first.exists() else None
if SITE is None:
    _obs, _own = load_config()
    SITE = next(s for s in all_sites(_own) if s.name == _obs.site_name)
print(f"Standort: {SITE.name} ({SITE.lat_deg:.3f}°, {SITE.lon_deg:.3f}°, {SITE.height_m:.0f} m)")
print(f"{MT['file'].nunique()} Bilder in der Messtabelle")

## Schritt 2 — Differenzphotometrie

Nullpunkt je Bild aus den Comps (Gaia/VSP-V), robust mit Ausreißer-Verwurf; Fehler aus Photonen/Himmel/Ausleserauschen
und Streuung der Comps. Die **Szintillation** (`scint_mag`, Young 1967) wird quadratisch zum Fehler des Ziels addiert und steht zusätzlich als eigene Spalte. Gesättigte Ziel-Pixel bleiben als `target_sat` markiert.

In [ ]:
CORE = "flux_ring" in MT and MT["flux_ring"].notna().any()      # Kernmaske aus NB04 vorhanden?
LC = light_curve(MT, STARS, "target", site=SITE, aperture_m=APERTURE_M, core_correct=CORE)
print(f"{len(LC)} Lichtkurvenpunkte aus {MT['file'].nunique()} Bildern; {int(LC['target_sat'].sum())} mit gesättigtem Ziel, "
      f"{int(LC['core_corrected'].sum())} davon mit Kernmaske korrigiert")
print(f"Magnitude: {LC['mag'].min():.2f} … {LC['mag'].max():.2f}  (Amplitude {LC['mag'].max() - LC['mag'].min():.2f} mag)")
if "check_minus_cat" in LC:
    sd = LC["check_minus_cat"].std()
    print(f"Check-Stern: Streuung {sd:.3f} mag, mittlere Abweichung vom Katalog {LC['check_minus_cat'].mean():+.3f} mag")
if "scint_mag" in LC:
    print(f"Szintillation (Young) je Bild: {LC['scint_mag'].median():.3f} mag; mittlerer Fehler je Punkt {LC['mag_err'].median():.3f} mag")
    print(f"Streuung der Comp-Nullpunkte {LC['zp_scatter'].median():.3f} mag — enthält Katalogfehler der Gaia-V (rote Comps) und ist kein reines Rauschmaß")
LC.head()

## Schritt 3 — Lomb-Scargle-Periode

Gesucht ist die Pulsationsperiode von δ Cep (Literatur 5,366 d). Die Frequenzauflösung ist durch die Beobachtungsdauer
begrenzt (1/T); die Periode gilt daher nur auf etwa diesen Rahmen genau.

In [ ]:
t = LC["bjd_tdb"].to_numpy(); y = LC["mag"].to_numpy()
freq = np.linspace(1 / 20, 1 / 1.0, 40000)
power = LombScargle(t, y).power(freq)
P = 1 / freq[np.argmax(power)]
T_span = t.max() - t.min()
print(f"Lomb-Scargle-Periode: {P:.3f} d  (Literatur 5,366 d, Abweichung {100 * (P / 5.366 - 1):+.1f} %)")
print(f"Zeitbasis {T_span:.1f} d → Frequenzauflösung ~ {T_span and 100 / T_span / (1 / P):.0f} % der Periode je 1/T")
from varphot import harmonic_fit
HF = harmonic_fit(t, y, 5.0, 5.8)
print(f"Fourier-Anpassung (3 Harmonische): Periode {HF['period']:.3f} d, Amplitude {HF['amplitude']:.2f} mag, Residuum {HF['rms']:.3f} mag")
plt.figure(figsize=(7, 2.6)); plt.plot(1 / freq, power); plt.axvline(5.366, color="r", ls=":", label="Literatur 5,366 d")
plt.xscale("log"); plt.xlabel("Periode [d]"); plt.ylabel("LS-Leistung"); plt.legend(); plt.show()

## Schritt 4 — Lichtkurve und Phasenkurve

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].errorbar(t - t.min(), y, LC["mag_err"], fmt=".", ms=4, color="C0", alpha=0.8)
s = LC["target_sat"].to_numpy()
ax[0].plot((t - t.min())[s], y[s], "o", mfc="none", mec="C3", ms=6, label="Ziel gesättigt")
ax[0].invert_yaxis(); ax[0].set_xlabel("Tage seit Beginn (BJD_TDB)"); ax[0].set_ylabel("V [mag]"); ax[0].legend(); ax[0].set_title("δ Cep — Lichtkurve")
ph = ((t - t.min()) / P) % 1
ax[1].plot(np.r_[ph, ph + 1], np.r_[y, y], ".", ms=4); ax[1].invert_yaxis()
ax[1].set_xlabel(f"Phase (P = {P:.3f} d)"); ax[1].set_ylabel("V [mag]"); ax[1].set_title("Phasenkurve")
plt.tight_layout(); plt.show()

## Schritt 5 — Export

In [ ]:
LC.to_csv(RESULTS / "zeitreihe.csv", index=False)
print("gespeichert:", RESULTS / "zeitreihe.csv")

## Schritt 6 — Transformation anwenden (optional, Farbkameras und RGB-Filter)

Nur mit Koeffizienten aus **NB08** (`VARPHOT_TRANSFORM_FILE`) und den Messtabellen der **drei Kanäle** (`VARPHOT_TAG_B`, `VARPHOT_TAG_G`, `VARPHOT_TAG_R`, je ein NB04-Lauf pro Kanal). Je Bild und Comp wird das Ziel transformiert
(`V_var = Δv + Tv_bv·Δ(B−V) + V_comp`, `Δ(B−V) = Tbv·Δ(b−g)`, Handbuch Gl. 6.11–6.14) und über die Comps gemittelt. Comps brauchen **AAVSO-Werte** für B und V (Gaia-abgeleitete werden abgelehnt). Das Ergebnis `zeitreihe_transformiert.csv` liest NB06.

In [ ]:
TRANSFORM_FILE = os.environ.get("VARPHOT_TRANSFORM_FILE")
if not TRANSFORM_FILE:
    print("Keine Transformation (VARPHOT_TRANSFORM_FILE nicht gesetzt) — Zeitreihe bleibt untransformiert (V bzw. TG/TB/TR).")
else:
    from varphot import TransformSolution, transformed_light_curve
    SOL = TransformSolution.load(TRANSFORM_FILE)
    base = RESULTS.parent
    CH = {"b": base / os.environ["VARPHOT_TAG_B"] / "messtabelle.csv", "g": base / os.environ["VARPHOT_TAG_G"] / "messtabelle.csv", "r": base / os.environ["VARPHOT_TAG_R"] / "messtabelle.csv"}
    MEAS = {ch: read_measurements(f) for ch, f in CH.items() if f.exists()}
    STARS_T = stars_from_field_config(record)
    STARS_T["B"] = [next((s["B"] for s in record["sterne"] if str(s["label"]) == i), np.nan) for i in STARS_T["id"]]
    STARS_T["R"] = [next((s.get("R") for s in record["sterne"] if str(s["label"]) == i), np.nan) for i in STARS_T["id"]]
    STARS_T["source"] = ["Gaia" if str(next((s["auid"] for s in record["sterne"] if str(s["label"]) == i), "")).startswith("Gaia") else "AAVSO" for i in STARS_T["id"]]
    LC_TR = transformed_light_curve(MEAS, STARS_T, "target", SOL)
    LC_TR.to_csv(RESULTS / "zeitreihe_transformiert.csv", index=False)
    print(f"transformiert: {len(LC_TR)} Bilder, V {LC_TR['V'].min():.2f}…{LC_TR['V'].max():.2f}, mittlere Streuung {LC_TR['V_err'].median():.3f} mag; gespeichert: {RESULTS / 'zeitreihe_transformiert.csv'}")
    print(f"Gültig im Farbbereich B−V {SOL.bv_range[0]:.2f}…{SOL.bv_range[1]:.2f} (Standardsterne); Ziel bei B−V ≈ {LC_TR['BV'].median():.2f}")
    if not (SOL.bv_range[0] <= LC_TR["BV"].median() <= SOL.bv_range[1]):
        print("⚠ Farbe des Ziels außerhalb des Bereichs der Standardsterne — Transformation unsicher")